In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Select a GPU runtime in Colab.")

Device: cuda
GPU: Tesla T4


In [2]:
!pip -q install diffusers transformers accelerate safetensors gradio


In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
import torch
import diffusers
import transformers
from diffusers import StableDiffusionPipeline

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())
print("StableDiffusionPipeline imported successfully!")

PyTorch: 2.11.0+cu130
Diffusers: 0.41.0
Transformers: 5.20.2
GPU available: True
StableDiffusionPipeline imported successfully!


In [2]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Select a GPU runtime in Colab."
    )

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")
pipe.enable_attention_slicing()

print("Stable Diffusion model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Stable Diffusion model loaded successfully!


In [ ]:
import gradio as gr
import zipfile
import os
import uuid
import torch

# PDF मधील original styles
styles = {
    "Realistic": "realistic, ultra detailed, 8k",
    "Anime": "anime style, vibrant colors",
    "Oil Painting": "oil painting, masterpiece",
    "Watercolor": "watercolor painting",
    "Pixel Art": "pixel art"
}

# Store generated files in a separate folder
OUTPUT_DIR = "/content/generated_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def generate_images(prompt, style, negative_prompt, num_images):
    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a text prompt first.")

    if style not in styles:
        raise gr.Error("Please select a valid image style.")

    num_images = int(num_images)

    if num_images < 1 or num_images > 4:
        raise gr.Error("Choose between 1 and 4 images.")

    final_prompt = prompt.strip() + ", " + styles[style]

    try:
        with torch.inference_mode():
            images = pipe(
                prompt=final_prompt,
                negative_prompt=negative_prompt or "",
                num_images_per_prompt=num_images
            ).images

        # Unique filenames prevent old images being overwritten
        batch_id = uuid.uuid4().hex[:8]
        image_files = []

        for i, img in enumerate(images, start=1):
            filename = os.path.join(
                OUTPUT_DIR,
                f"generated_{batch_id}_{i}.png"
            )
            img.save(filename)
            image_files.append(filename)

        # Create a ZIP of all images from this generation
        zip_filename = os.path.join(
            OUTPUT_DIR, f"images_{batch_id}.zip"
        )

        with zipfile.ZipFile(
            zip_filename, "w", zipfile.ZIP_DEFLATED
        ) as zipf:
            for file in image_files:
                zipf.write(file, arcname=os.path.basename(file))

        return images, zip_filename

    except gr.Error:
        raise
    except Exception as e:
        raise gr.Error(
            f"Image generation failed: {str(e)[:300]}"
        )


# Custom professional UI
custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}
#hero {
    text-align: center;
    padding: 28px 16px;
    border-radius: 18px;
    background: linear-gradient(120deg, #111827, #312e81, #6d28d9);
    color: white;
    margin-bottom: 18px;
}
#hero h1 {
    font-size: 36px;
    margin-bottom: 8px;
}
#hero p {
    font-size: 16px;
    opacity: 0.92;
}
#generate {
    background: linear-gradient(90deg, #4f46e5, #9333ea);
    color: white;
    border: none;
    font-weight: bold;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css,
    title="GenAI Studio | Experiment 4"
) as demo:

    gr.HTML("""
    <div id="hero">
        <h1>✦ GenAI Studio</h1>
        <p>Experiment 4 · AI Image Generation with Stable Diffusion</p>
        <p>Turn your imagination into images using text prompts.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ✍️ Create your image")

            prompt = gr.Textbox(
                label="Image Prompt",
                placeholder=(
                    "e.g. A flock of colorful birds "
                    "flying over a misty mountain lake"
                ),
                lines=4
            )

            style = gr.Dropdown(
                choices=list(styles.keys()),
                value="Realistic",
                label="Choose an Art Style"
            )

            negative_prompt = gr.Textbox(
                label="Negative Prompt (Optional)",
                placeholder=(
                    "e.g. blurry, low quality, distorted, "
                    "extra limbs"
                ),
                lines=2
            )

            num_images = gr.Slider(
                minimum=1,
                maximum=4,
                step=1,
                value=1,
                label="Number of Images"
            )

            generate_btn = gr.Button(
                "✨ Generate Images",
                variant="primary",
                elem_id="generate"
            )

            gr.Markdown(
                "**Tip:** Describe the subject, lighting, "
                "background and mood for better results."
            )

        with gr.Column(scale=1):
            gr.Markdown("### 🖼️ Your creations")

            gallery = gr.Gallery(
                label="Generated Image Gallery",
                columns=2,
                height=420,
                object_fit="contain",
                show_label=True
            )

            download = gr.File(
                label="Download all images as ZIP"
            )

    gr.Examples(
        examples=[
            ["A beautiful peacock in a lush green garden", "Realistic"],
            ["A futuristic city at sunset", "Anime"],
            ["A peaceful village beside a river", "Watercolor"],
            ["A fantasy castle among clouds", "Oil Painting"],
            ["A tiny pixel art house in a forest", "Pixel Art"]
        ],
        inputs=[prompt, style],
        label="Try these example prompts"
    )

    gr.Markdown(
        "---\n"
        "GenAI Studio · Built with PyTorch, Diffusers and Gradio"
    )

    # PDF मधील button-to-function integration
    generate_btn.click(
        fn=generate_images,
        inputs=[
            prompt,
            style,
            negative_prompt,
            num_images
        ],
        outputs=[
            gallery,
            download
        ]
    )

demo.queue().launch(share=True, debug=True)

/tmp/ipykernel_2098/1499407692.py:106: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2403efb86867929c98.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]